In [ ]:
import gi
gi.require_version('Gtk', '3.0')
from gi.repository import Gtk, GLib, Gdk, Pango

import threading
import os
import re
import hashlib
import bs4
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.llms import Ollama
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain.chains import create_retrieval_chain

class ChatbotWindow(Gtk.Window):
    def __init__(self):
        Gtk.Window.__init__(self, title="RAG Chatbot")
        self.set_default_size(500, 400)
        self.set_border_width(10)

        self.apply_css()

        vbox = Gtk.Box(orientation=Gtk.Orientation.VERTICAL, spacing=10)
        self.add(vbox)

        action_bar = Gtk.Box(orientation=Gtk.Orientation.HORIZONTAL, spacing=5)
        vbox.pack_start(action_bar, False, False, 0)

        clear_button = Gtk.Button.new_from_icon_name("edit-clear-all-symbolic", Gtk.IconSize.MENU)
        clear_button.set_tooltip_text("Clear conversation")
        clear_button.connect("clicked", self.on_clear_button_clicked)
        action_bar.pack_start(clear_button, False, False, 0)
        
        save_button = Gtk.Button.new_from_icon_name("document-save-symbolic", Gtk.IconSize.MENU)
        save_button.set_tooltip_text("Save conversation")
        save_button.connect("clicked", self.on_save_button_clicked)
        action_bar.pack_start(save_button, False, False, 0)
        
        file_button = Gtk.Button.new_from_icon_name("document-open-symbolic", Gtk.IconSize.MENU)
        file_button.set_tooltip_text("Select Knowledge Base (PDF)")
        file_button.connect("clicked", self.on_file_button_clicked)
        action_bar.pack_start(file_button, False, False, 0)

        scrolled_window = Gtk.ScrolledWindow()
        scrolled_window.set_hexpand(True)
        scrolled_window.set_vexpand(True)
        vbox.pack_start(scrolled_window, True, True, 0)
        scrolled_window.get_style_context().add_class("chat-history")

        self.message_view = Gtk.TextView()
        self.message_view.set_editable(False)
        self.message_view.set_cursor_visible(False)
        self.message_view.set_left_margin(10)
        self.message_view.set_right_margin(10)
        self.message_view.set_wrap_mode(Gtk.WrapMode.WORD)
        scrolled_window.add(self.message_view)

        self.message_buffer = self.message_view.get_buffer()
        self.tag_user = self.message_buffer.create_tag(
            "user", 
            foreground="#333333", 
            weight=600, 
            justification=Gtk.Justification.LEFT
        )
        self.tag_chatbot = self.message_buffer.create_tag(
            "chatbot", 
            foreground="#00796b", 
            weight=400, 
            justification=Gtk.Justification.LEFT
        )
        self.tag_system = self.message_buffer.create_tag(
            "system", 
            foreground="#999999", 
            style=Pango.Style.ITALIC, 
            justification=Gtk.Justification.CENTER
        )
        
        input_box = Gtk.Box(orientation=Gtk.Orientation.HORIZONTAL, spacing=6)
        vbox.pack_start(input_box, False, False, 0)

        self.input_entry = Gtk.Entry()
        self.input_entry.set_hexpand(True)
        self.input_entry.get_style_context().add_class("input-entry")
        input_box.pack_start(self.input_entry, True, True, 0)
        self.input_entry.connect("activate", self.on_send_button_clicked)

        self.send_button = Gtk.Button.new_from_icon_name("mail-send-receive-symbolic", Gtk.IconSize.MENU)
        self.send_button.set_tooltip_text("Send message")
        self.send_button.connect("clicked", self.on_send_button_clicked)
        input_box.pack_start(self.send_button, False, False, 0)

        self.spinner = Gtk.Spinner()
        input_box.pack_start(self.spinner, False, False, 0)

        self.retrieval_chain = None
        self.current_pdf_path = None
        self.append_message("Select a PDF to begin.", "system")
        self.input_entry.set_sensitive(False)
        self.send_button.set_sensitive(False)

    def apply_css(self):
        style_provider = Gtk.CssProvider()
        try:
            style_provider.load_from_path('style.css')
            Gtk.StyleContext.add_provider_for_screen(
                Gdk.Screen.get_default(),
                style_provider,
                Gtk.STYLE_PROVIDER_PRIORITY_APPLICATION
            )
        except Exception as e:
            print(f"Failed to load CSS: {e}")

    def on_file_button_clicked(self, widget):
        dialog = Gtk.FileChooserDialog(
            "Please choose a PDF file", self, Gtk.FileChooserAction.OPEN,
            (Gtk.STOCK_CANCEL, Gtk.ResponseType.CANCEL, "Select", Gtk.ResponseType.ACCEPT)
        )
        
        filter_pdf = Gtk.FileFilter()
        filter_pdf.set_name("PDF files")
        filter_pdf.add_mime_type("application/pdf")
        dialog.add_filter(filter_pdf)
        
        response = dialog.run()
        if response == Gtk.ResponseType.ACCEPT:
            pdf_path = dialog.get_filename()
            self.current_pdf_path = pdf_path
            self.input_entry.set_sensitive(False)
            self.send_button.set_sensitive(False)
            self.append_message(f"Loading {os.path.basename(pdf_path)}...", "system")
            self.spinner.start()
            
            thread = threading.Thread(target=self.initialize_chatbot_logic)
            thread.daemon = True
            thread.start()
            
        dialog.destroy()

    def get_document_signature(self, file_path):
        """Generates a hash of the PDF file's content."""
        hasher = hashlib.sha256()
        try:
            with open(file_path, 'rb') as f:
                while chunk := f.read(4096):
                    hasher.update(chunk)
            return hasher.hexdigest()
        except Exception:
            return None

    def initialize_chatbot_logic(self):
        try:
            persist_directory = "./chroma_db"
            os.makedirs(persist_directory, exist_ok=True)
            signature_file = os.path.join(persist_directory, "doc_signature.txt")
            current_signature = self.get_document_signature(self.current_pdf_path)

            load_from_disk = False
            if os.path.exists(signature_file):
                with open(signature_file, 'r') as f:
                    saved_signature = f.read()
                if saved_signature == current_signature:
                    load_from_disk = True

            if load_from_disk:
                GLib.idle_add(self.append_message, "Loading existing embeddings...", "system")
                embeddings = OllamaEmbeddings(model="nomic-embed-text")
                vector_store = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
            else:
                if os.path.exists(persist_directory) and os.listdir(persist_directory):
                    GLib.idle_add(self.ask_reprocess_dialog)
                    return # Wait for user input

                GLib.idle_add(self.append_message, "Creating new embeddings...", "system")
                loader = PyPDFLoader(self.current_pdf_path)
                docs = loader.load()
                text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
                all_splits = text_splitter.split_documents(docs)
                embeddings = OllamaEmbeddings(model="nomic-embed-text")
                vector_store = Chroma.from_documents(documents=all_splits, embedding=embeddings, persist_directory=persist_directory)
                with open(signature_file, 'w') as f:
                    f.write(current_signature)

            llm = Ollama(model="gemma3:1b")
            retriever = vector_store.as_retriever()
            prompt = ChatPromptTemplate.from_template("""Answer the user's question based on the provided context:
<context>
{context}
</context>
Question: {input}""")
            document_chain = create_stuff_documents_chain(llm, prompt)
            self.retrieval_chain = create_retrieval_chain(retriever, document_chain)
            
            GLib.idle_add(self.append_message, "Backend ready. Ask me a question!", "system")
            GLib.idle_add(self.input_entry.set_sensitive, True)
            GLib.idle_add(self.send_button.set_sensitive, True)

        except Exception as e:
            GLib.idle_add(self.append_message, f"Failed to initialize backend. Error: {e}", "system")
        finally:
            GLib.idle_add(self.spinner.stop)
            return False

    def ask_reprocess_dialog(self):
        dialog = Gtk.MessageDialog(self, 0, Gtk.MessageType.QUESTION,
            Gtk.ButtonsType.YES_NO, "The document's signature has changed. Do you want to re-process it?")
        response = dialog.run()
        dialog.destroy()
        if response == Gtk.ResponseType.YES:
            # Re-process the document if the user confirms
            thread = threading.Thread(target=self.initialize_chatbot_logic_force_reprocess)
            thread.daemon = True
            thread.start()
        else:
            GLib.idle_add(self.append_message, "Using existing (potentially outdated) embeddings.", "system")
            self.input_entry.set_sensitive(True)
            self.send_button.set_sensitive(True)

    def initialize_chatbot_logic_force_reprocess(self):
        try:
            # Re-process logic (same as the 'else' block above)
            persist_directory = "./chroma_db"
            GLib.idle_add(self.append_message, "Creating new embeddings...", "system")
            loader = PyPDFLoader(self.current_pdf_path)
            docs = loader.load()
            text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
            all_splits = text_splitter.split_documents(docs)
            embeddings = OllamaEmbeddings(model="nomic-embed-text")
            vector_store = Chroma.from_documents(documents=all_splits, embedding=embeddings, persist_directory=persist_directory)
            with open(os.path.join(persist_directory, "doc_signature.txt"), 'w') as f:
                f.write(self.get_document_signature(self.current_pdf_path))
            
            # Continue with chain building
            llm = Ollama(model="gemma3:1b")
            retriever = vector_store.as_retriever()
            prompt = ChatPromptTemplate.from_template("""Answer the user's question based on the provided context:
<context>
{context}
</context>
Question: {input}""")
            document_chain = create_stuff_documents_chain(llm, prompt)
            self.retrieval_chain = create_retrieval_chain(retriever, document_chain)
            
            GLib.idle_add(self.append_message, "Backend ready. Ask me a question!", "system")
            GLib.idle_add(self.input_entry.set_sensitive, True)
            GLib.idle_add(self.send_button.set_sensitive, True)

        except Exception as e:
            GLib.idle_add(self.append_message, f"Failed to initialize backend. Error: {e}", "system")
        finally:
            GLib.idle_add(self.spinner.stop)

    def on_send_button_clicked(self, widget):
        user_text = self.input_entry.get_text()
        if not user_text:
            return

        self.append_message(user_text, "user")
        self.input_entry.set_text("")
        self.input_entry.set_sensitive(False)
        self.send_button.set_sensitive(False)
        self.spinner.start()

        thread = threading.Thread(target=self.get_and_display_response, args=(user_text,))
        thread.daemon = True
        thread.start()

    def get_and_display_response(self, user_text):
        try:
            if not self.retrieval_chain:
                GLib.idle_add(self.append_message, "Please select a document first.", "system")
                return
            
            response = self.retrieval_chain.invoke({"input": user_text})
            chatbot_response = response['answer']
            
            formatted_response = self.format_response(chatbot_response)
            
            GLib.idle_add(self.append_message, formatted_response, "chatbot")
        
        except Exception as e:
            GLib.idle_add(self.append_message, f"An error occurred: {e}", "system")
            
        finally:
            GLib.idle_add(self.spinner.stop)
            GLib.idle_add(self.input_entry.set_sensitive, True)
            GLib.idle_add(self.send_button.set_sensitive, True)

    def format_response(self, text):
        text = re.sub(r'\n{2,}', '\n', text)
        text = re.sub(r'(\d+\.)\s', r'\n\1 ', text)
        text = re.sub(r'([-*])\s', r'\n\1 ', text)
        text = re.sub(r'\*\*(.*?)\*\*', r'<b>\1</b>', text) 
        
        return text

    def on_clear_button_clicked(self, widget):
        self.message_buffer.set_text("")
        self.append_message("Chat history cleared.", "system")

    def on_save_button_clicked(self, widget):
        dialog = Gtk.FileChooserDialog(
            "Save Conversation", self, Gtk.FileChooserAction.SAVE,
            (Gtk.STOCK_CANCEL, Gtk.ResponseType.CANCEL, "Save", Gtk.ResponseType.ACCEPT)
        )
        dialog.set_current_name("chatbot_conversation.txt")

        response = dialog.run()
        if response == Gtk.ResponseType.ACCEPT:
            filename = dialog.get_filename()
            with open(filename, "w") as f:
                start_iter, end_iter = self.message_buffer.get_bounds()
                text = self.message_buffer.get_text(start_iter, end_iter, True)
                f.write(text)
        
        dialog.destroy()

    def append_message(self, message, tag_name):
        end_iter = self.message_buffer.get_end_iter()
        start_iter = self.message_buffer.get_end_iter()
        
        if tag_name == "user":
            self.message_buffer.insert(end_iter, "You: ")
        elif tag_name == "chatbot":
            self.message_buffer.insert(end_iter, "Chatbot: ")
        elif tag_name == "system":
            self.message_buffer.insert(end_iter, "System: ")

        self.message_buffer.insert(end_iter, message + "\n")
        
        end_iter = self.message_buffer.get_end_iter()
        self.message_buffer.apply_tag_by_name(tag_name, start_iter, end_iter)
        
        self.message_view.scroll_to_iter(end_iter, 0.0, True, 0.0, 1.0)
        return False

if __name__ == "__main__":
    win = ChatbotWindow()
    win.connect("destroy", Gtk.main_quit)
    win.show_all()
    Gtk.main()


(ipykernel_launcher.py:161330): Gtk-WARNING **: 20:28:54.889: Invalid text buffer iterator: either the iterator is uninitialized, or the characters/pixbufs/widgets in the buffer have been modified since the iterator was created.
You must use marks, character numbers, or line numbers to preserve a position across buffer modifications.
You can apply tags and insert marks without invalidating your iterators,
but any mutation that affects 'indexable' buffer contents (contents that can be referred to by character offset)
will invalidate all outstanding iterators

(ipykernel_launcher.py:161330): Gtk-CRITICAL **: 20:28:54.889: gtk_text_buffer_apply_tag_by_name: assertion 'gtk_text_iter_get_buffer (start) == buffer' failed
/tmp/ipykernel_161330/1441084871.py:118: PyGTKDeprecationWarning: Using positional arguments with the GObject constructor has been deprecated. Please specify keyword(s) for "title, parent, action, buttons" or use a class specific constructor. See: https://wiki.gnome.org/PyG